code pour ajouter les variabls géopolitiques. vibe-codé AVEC VERIFICATION car pas le temps, et j'attendais ces variables depuis 1 mois.

* intensité guerre: OK (hyper-reg)
* regime politique (de -10 à +10): Polity5 ou VDem (hyper-reg)
* distance en niveau de regime politiqye (dyadique) 
* conflit partagé (dyadique) 

* ATTENTION: ces variables doivent inclure les chocs à t, t-1 an.


In [22]:
import pandas as pd
import numpy as np
import country_converter as coco

In [23]:
df_main = pd.read_csv('/Users/romain/Desktop/Projets DS/ProjetStat/data/FINAL_GRAVITY_TRAINING_MATRIX.csv')
df_main.columns

Index(['orig', 'dest', 'iso3_d', 'year', 'iso3_o', 'flow', 'P_it', 'PSR_i',
       'IMR_it', 'urban_it', 'LA_i', 'LL_i', 'P_jt', 'PSR_j', 'IMR_jt',
       'urban_jt', 'LA_j', 'LL_j', 'D_ij', 'LB_ij', 'OL_ij', 'COL_ij',
       't_2000', 't_2000_sq', 'flow_raw', 'log_flow_plus_1', 'ihs_flow',
       'is_migration', 'gdp_o', 'gdpcap_o', 'gdp_d', 'gdpcap_d', 'gdp_o_lag',
       'gdpcap_o_lag', 'gdp_d_lag', 'gdpcap_d_lag', 'log_gdp_o', 'log_gdp_d',
       'log_gdpcap_o', 'log_gdpcap_d', 'log_gdp_o_lag', 'log_gdp_d_lag',
       'log_gdpcap_o_lag', 'log_gdpcap_d_lag'],
      dtype='object')

In [24]:

df_UCDP = pd.read_csv('/Users/romain/Desktop/geopo/UcdpPrioConflict_v25_1.csv')
df_V_Dem = pd.read_csv('/Users/romain/Desktop/geopo/V-Dem-CY-Core-v16.csv')



In [ ]:

# V-DEM : Extraction et Lag (t-1)

vdem_cols = ['country_text_id', 'year', 'v2x_polyarchy', 'v2x_clphy']
df_vdem_sub = df_V_Dem[vdem_cols].copy()

df_vdem_sub['year_merge'] = df_vdem_sub['year'] + 1 
df_vdem_sub = df_vdem_sub.drop(columns=['year'])
df_vdem_sub = df_vdem_sub.rename(columns={'country_text_id': 'iso3'})

#UCDP : Désagrégation spatiale, GWcode -> ISO3 et Lag (t-1)

df_ucdp_sub = df_UCDP[['gwno_loc', 'year', 'intensity_level', 'type_of_conflict']].copy()

# Séparation des codes GWNO multiples
df_ucdp_sub['gwno_loc'] = df_ucdp_sub['gwno_loc'].astype(str).str.split(',')
df_ucdp_sub = df_ucdp_sub.explode('gwno_loc')
df_ucdp_sub['gwno_loc'] = df_ucdp_sub['gwno_loc'].str.strip()

# Typage strict obligatoire pour la lecture du GWcode par countryconverter
df_ucdp_sub['gwno_loc'] = pd.to_numeric(df_ucdp_sub['gwno_loc'], errors='coerce')
df_ucdp_sub = df_ucdp_sub.dropna(subset=['gwno_loc'])

# Conversion (Correction 'GWcode')
cc = coco.CountryConverter()
df_ucdp_sub['iso3'] = cc.convert(names=df_ucdp_sub['gwno_loc'].tolist(), src='GWcode', to='ISO3', not_found=np.nan)
df_ucdp_sub = df_ucdp_sub.dropna(subset=['iso3'])

df_ucdp_agg = df_ucdp_sub.groupby(['iso3', 'year'])[['intensity_level', 'type_of_conflict']].max().reset_index()

df_ucdp_agg['year_merge'] = df_ucdp_agg['year'] + 1
df_ucdp_agg = df_ucdp_agg.drop(columns=['year'])

# Fusion Monadique (Country, Year, Transitoire! pour propreté)

df_geo = pd.merge(df_vdem_sub, df_ucdp_agg, on=['iso3', 'year_merge'], how='left')

df_geo['intensity_level'] = df_geo['intensity_level'].fillna(0)
df_geo['type_of_conflict'] = df_geo['type_of_conflict'].fillna(0)

# Génération de la base df_main_transitoire (Jointure Dyadique Double avec suffixe _lag)

df_main_transitoire = df_main[['orig', 'dest', 'year']].drop_duplicates().copy()

df_main_transitoire = pd.merge(
    df_main_transitoire, 
    df_geo.rename(columns={'iso3': 'orig', 'year_merge': 'year'}),
    on=['orig', 'year'], 
    how='left'
).rename(columns={
    'v2x_polyarchy': 'v2x_polyarchy_o_lag',
    'v2x_clphy': 'v2x_clphy_o_lag',
    'intensity_level': 'intensity_level_o_lag',
    'type_of_conflict': 'type_of_conflict_o_lag'
})

df_main_transitoire = pd.merge(
    df_main_transitoire, 
    df_geo.rename(columns={'iso3': 'dest', 'year_merge': 'year'}),
    on=['dest', 'year'], 
    how='left'
).rename(columns={
    'v2x_polyarchy': 'v2x_polyarchy_d_lag',
    'v2x_clphy': 'v2x_clphy_d_lag',
    'intensity_level': 'intensity_level_d_lag',
    'type_of_conflict': 'type_of_conflict_d_lag'
})

751 not found in GWcode
751 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
816 not found in GWcode
816 not found in GWcode
816 not found in GWcode
816 not found in GWcode
816 not found in GWcode
816 not found in GWcode
816 not found in GWcode
816 not found in GWcode
816 not found in GWcode
751 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in

In [27]:
df_main_transitoire.columns

Index(['orig', 'dest', 'year', 'v2x_polyarchy_o_lag', 'v2x_clphy_o_lag',
       'intensity_level_o_lag', 'type_of_conflict_o_lag',
       'v2x_polyarchy_d_lag', 'v2x_clphy_d_lag', 'intensity_level_d_lag',
       'type_of_conflict_d_lag'],
      dtype='object')

FUSION

In [29]:
# Fusion finale par les clés dyadiques temporelles
df_main = df_main.merge(df_main_transitoire, on=['orig', 'dest', 'year'], how='left')

# Imputation de sécurité pour les conflits
# Si une dyade n'a pas trouvé de correspondance, on présume l'absence de conflit (0)
conflict_cols = [
    'intensity_level_o_lag', 'type_of_conflict_o_lag',
    'intensity_level_d_lag', 'type_of_conflict_d_lag'
]
df_main[conflict_cols] = df_main[conflict_cols].fillna(0)

# Pour V-Dem, on ne remplit pas par 0 (car 0 est une valeur de l'indice).
# On laisse les NaN s'il y en a pour les identifier lors de l'audit.

In [ ]:
# Tri spatio-temporel strict obligatoire pour garantir l'ordre du backfill
df_main = df_main.sort_values(by=['orig', 'dest', 'year'])

vdem_vars_o = ['v2x_polyarchy_o_lag', 'v2x_clphy_o_lag']
vdem_vars_d = ['v2x_polyarchy_d_lag', 'v2x_clphy_d_lag']

# Imputation Origine (bfill pour l'URSS, ffill par sécurité, puis médiane globale pour les micro-états)
for col in vdem_vars_o:
    # Rétro-projection intra-nœud
    df_main[col] = df_main.groupby('orig')[col].transform(lambda x: x.bfill().ffill())
    # Remplissage de dernier recours pour les micro-États ignorés par V-Dem
    df_main[col] = df_main[col].fillna(df_main[col].median())

# Imputation Destination
for col in vdem_vars_d:
    df_main[col] = df_main.groupby('dest')[col].transform(lambda x: x.bfill().ffill())
    df_main[col] = df_main[col].fillna(df_main[col].median())

# Audit de vérification finale (doit retourner 0 absolu)
print("NaN résiduels sur les variables géopolitiques :")
print(df_main[vdem_vars_o + vdem_vars_d].isna().sum())

NaN résiduels sur les variables géopolitiques :
v2x_polyarchy_o_lag    0
v2x_clphy_o_lag        0
v2x_polyarchy_d_lag    0
v2x_clphy_d_lag        0
dtype: int64


In [32]:
# Vérification rapide du taux de remplissage
geo_vars = [
    'v2x_polyarchy_o_lag', 'v2x_clphy_o_lag', 
    'v2x_polyarchy_d_lag', 'v2x_clphy_d_lag'
]

print("Audit des NaN après fusion :")
print(df_main[geo_vars + conflict_cols].isna().sum())

# Si des NaN persistent dans les variables V-Dem, identifiez les pays concernés :
missing_vdem = df_main[df_main['v2x_polyarchy_o_lag'].isna()]['orig'].unique()
if len(missing_vdem) > 0:
    print(f"\nPays sans données V-Dem : {missing_vdem}")

Audit des NaN après fusion :
v2x_polyarchy_o_lag       0
v2x_clphy_o_lag           0
v2x_polyarchy_d_lag       0
v2x_clphy_d_lag           0
intensity_level_o_lag     0
type_of_conflict_o_lag    0
intensity_level_d_lag     0
type_of_conflict_d_lag    0
dtype: int64


In [33]:
df_main.columns

Index(['orig', 'dest', 'iso3_d', 'year', 'iso3_o', 'flow', 'P_it', 'PSR_i',
       'IMR_it', 'urban_it', 'LA_i', 'LL_i', 'P_jt', 'PSR_j', 'IMR_jt',
       'urban_jt', 'LA_j', 'LL_j', 'D_ij', 'LB_ij', 'OL_ij', 'COL_ij',
       't_2000', 't_2000_sq', 'flow_raw', 'log_flow_plus_1', 'ihs_flow',
       'is_migration', 'gdp_o', 'gdpcap_o', 'gdp_d', 'gdpcap_d', 'gdp_o_lag',
       'gdpcap_o_lag', 'gdp_d_lag', 'gdpcap_d_lag', 'log_gdp_o', 'log_gdp_d',
       'log_gdpcap_o', 'log_gdpcap_d', 'log_gdp_o_lag', 'log_gdp_d_lag',
       'log_gdpcap_o_lag', 'log_gdpcap_d_lag', 'v2x_polyarchy_o_lag',
       'v2x_clphy_o_lag', 'intensity_level_o_lag', 'type_of_conflict_o_lag',
       'v2x_polyarchy_d_lag', 'v2x_clphy_d_lag', 'intensity_level_d_lag',
       'type_of_conflict_d_lag'],
      dtype='object')

In [34]:
df_main.to_csv("data_bayesian_arx_hurdle.csv", index=False)